## Retry and Delay

In [1]:
import os
import time
import logging
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("api_key")

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Initialize LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=api_key)

def call_llm_with_retry(prompt, retries=3, delay=2):
    for attempt in range(1, retries + 1):
        try:
            logging.info(f"Attempt {attempt} - Calling LLM")
            
            response = llm.invoke([HumanMessage(content=prompt)])
            
            logging.info("LLM call successful")
            return response.content
        
        except Exception as e:
            logging.error(f"Attempt {attempt} failed: {e}")
            
            if attempt < retries:
                time.sleep(delay)
            else:
                logging.critical("All retry attempts failed")
                raise

# Example usage
if __name__ == "__main__":
    prompt = "Explain what is an API in one sentence"
    
    result = call_llm_with_retry(prompt)
    print("Final Response:", result)

/Users/ishant162/miniconda3/envs/rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-18 09:59:54,491 - INFO - Attempt 1 - Calling LLM
2026-04-18 09:59:54,804 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-18 09:59:54,811 - INFO - LLM call successful


Final Response: An Application Programming Interface (API) is a set of defined rules that enables different software systems, applications, or services to communicate and exchange data with each other in a standardized and secure manner.


## Simple RAG

In [2]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
from pathlib import Path
import uuid
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

# ── Config ────────────────────────────────────────────────────────────────────
PDF_DIR        = "./data/"
VECTOR_DIR     = "./data/vector_store"
EMBED_MODEL    = "all-MiniLM-L6-v2"
CHUNK_SIZE     = 1000
CHUNK_OVERLAP  = 200
TOP_K          = 3

# ── 1. Load PDFs ──────────────────────────────────────────────────────────────
docs = []
for pdf in Path(PDF_DIR).glob("**/*.pdf"):
    docs.extend(PyPDFLoader(str(pdf)).load())
print(f"Loaded {len(docs)} pages")

# ── 2.1 Fixed Chunk ──────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

# ── 2.2 Semantic Chunk ──────────────────────────────────────────────────────────────────

# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# splitter = SemanticChunker(
#     embeddings=embeddings,
#     breakpoint_threshold_type="percentile",
#     breakpoint_threshold_amount=70
# )
# chunks = splitter.split_documents(docs)
# print(f"Split into {len(chunks)} semantic chunks")

# ── 3. Embed ──────────────────────────────────────────────────────────────────
model = SentenceTransformer(EMBED_MODEL)
texts = [c.page_content for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

# ── 4. Store in ChromaDB ──────────────────────────────────────────────────────
os.makedirs(VECTOR_DIR, exist_ok=True)

# Check write access
if not os.access(VECTOR_DIR, os.W_OK):
    raise PermissionError(f"No write permission for {VECTOR_DIR}")

client     = chromadb.PersistentClient(path=VECTOR_DIR)
collection = client.get_or_create_collection("pdf_documents")

collection.add(
    ids        = [f"doc_{uuid.uuid4().hex[:8]}_{i}" for i in range(len(chunks))],
    embeddings = embeddings.tolist(),
    documents  = texts,
    metadatas  = [c.metadata for c in chunks]
)
print(f"Stored {collection.count()} chunks in vector store")


2026-04-18 09:59:56,670 - INFO - Use pytorch device_name: mps
2026-04-18 09:59:56,670 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Loaded 15 pages
Split into 52 chunks


Batches: 100%|██████████| 2/2 [00:00<00:00,  3.47it/s]
2026-04-18 10:00:01,210 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Stored 52 chunks in vector store


In [3]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ── Shared setup (reuse model + collection from the main pipeline) ─────────────
texts        = [c.page_content for c in chunks]   # already have this from pipeline
tokenized    = [t.lower().split() for t in texts]
bm25         = BM25Okapi(tokenized)
reranker     = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# ── BM25 search ───────────────────────────────────────────────────────────────
def bm25_search(query, top_k):
    scores     = bm25.get_scores(query.lower().split())
    top_idx    = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"id": f"bm25_{i}", "content": texts[i], "score": scores[i]} for i in top_idx if scores[i] > 0]

# ── Vector search ─────────────────────────────────────────────────────────────
def vector_search(query, top_k):
    q_emb   = model.encode([query])[0].tolist()   # reuse SentenceTransformer from pipeline
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    return [
        {"id": doc_id, "content": doc, "score": 1 - dist}
        for doc_id, doc, dist in zip(results["ids"][0], results["documents"][0], results["distances"][0])
    ]

# ── Reciprocal Rank Fusion ────────────────────────────────────────────────────
def rrf(bm25_results, vector_results, k=60):
    scores, doc_map = {}, {}
    for rank, doc in enumerate(bm25_results + vector_results, 1):
        scores[doc["id"]]  = scores.get(doc["id"], 0) + 1 / (k + rank)
        doc_map[doc["id"]] = doc
    return [doc_map[i] for i in sorted(scores, key=lambda x: scores[x], reverse=True)]

# ── RAG variants ──────────────────────────────────────────────────────────────
def rag(query, top_k=3):
    """Plain vector RAG"""
    results = vector_search(query, top_k)
    return _generate(query, results)

def rag_hybrid(query, top_k=3):
    """BM25 + Vector fused via RRF"""
    results = rrf(bm25_search(query, top_k * 3), vector_search(query, top_k * 3))[:top_k]
    return _generate(query, results)

def rag_hybrid_rerank(query, top_k=3):
    """BM25 + Vector → RRF → CrossEncoder rerank"""
    candidates = rrf(bm25_search(query, top_k * 4), vector_search(query, top_k * 4))[:top_k * 2]
    scores     = reranker.predict([(query, d["content"]) for d in candidates])
    results    = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return _generate(query, [d for d, _ in results])

def rag_hyde(query, top_k=3):
    """HyDE: generate hypothetical answer → use it for retrieval"""
    hypo    = llm.invoke([f"Write a concise answer to: {query}"]).content
    results = rrf(bm25_search(hypo, top_k * 3), vector_search(hypo, top_k * 3))[:top_k]
    return _generate(query, results)

# ── Shared answer generator ───────────────────────────────────────────────────
def _generate(query, results):
    context = "\n\n".join([d["content"] for d in results])
    if not context:
        return "No relevant context found."
    prompt = f"Use the context below to answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    return llm.invoke([prompt]).content

# ── Run ───────────────────────────────────────────────────────────────────────
q = "What is attention is all you need?"

print("*"*50)
print("Basic RAG")
print(rag(q))
print("*"*50)
print("Hybrid RAG")
print(rag_hybrid(q))
print("*"*50)
print("Hybrid RAG rerank")
print(rag_hybrid_rerank(q))
print("*"*50)
print("Hyde RAG")
print(rag_hyde(q))
print("*"*50)

2026-04-18 10:00:04,018 - INFO - Use pytorch device: mps


**************************************************
Basic RAG


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.84it/s]
2026-04-18 10:00:05,792 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The context provided doesn't directly answer the question "What is attention is all you need?" However, based on the information given, it seems to be referring to the concept of attention in the context of neural networks, specifically transformers.

In the paper "Attention is All You Need" by Vaswani et al., the authors propose a new neural network architecture called the Transformer, which relies entirely on self-attention mechanisms to process input sequences. The Transformer model replaces traditional recurrent neural network (RNN) and convolutional neural network (CNN) architectures with a purely attention-based approach.

In essence, the idea is that attention mechanisms can be used to weigh the importance of different input elements relative to each other, allowing the model to focus on the most relevant information when making predictions. This approach has been shown to be highly effective in various natural language processing tasks, such as machine translation, text classif

Batches: 100%|██████████| 1/1 [00:00<00:00, 113.59it/s]
2026-04-18 10:00:06,619 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


"Attention Is All You Need" is the title of a research paper by Ashish Vaswani, Noam Shazeer, Niki Parmar, and others, published by Google. The paper proposes a new simple network architecture called the Transformer, which is based solely on attention mechanisms, dispensing with recurrence and convolutions. In other words, the paper argues that attention mechanisms are sufficient for sequence transduction models, and that they can replace more complex recurrent or convolutional neural networks.
**************************************************
Hybrid RAG rerank


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.94it/s]
2026-04-18 10:00:07,441 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


"Attention Is All You Need" is the title of a research paper by Ashish Vaswani et al., which proposes a new network architecture called the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions. In essence, the paper argues that attention mechanisms are sufficient to handle sequence transduction tasks, and that complex recurrent or convolutional neural networks are not necessary. The paper presents a simple and efficient architecture that relies entirely on self-attention mechanisms to process input sequences, and demonstrates its effectiveness in various tasks.
**************************************************
Hyde RAG


2026-04-18 10:00:07,977 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.28it/s]
2026-04-18 10:00:08,865 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


"Attention Is All You Need" is the title of a research paper that proposes a new network architecture called the Transformer, which is based solely on attention mechanisms and dispenses with recurrence and convolutions. In essence, the paper argues that attention mechanisms are sufficient to handle sequence transduction tasks, such as machine translation, and can outperform traditional recurrent neural network (RNN) and convolutional neural network (CNN) architectures. The Transformer model uses self-attention mechanisms to relate different positions of a single sequence and compute a representation of the sequence, allowing it to learn dependencies between distant positions more efficiently.
**************************************************


## RAG with Pinecone

In [4]:
########### Setting up Pinecone
from pinecone import Pinecone

pinecone_api_key = os.getenv("pinecone_api_key")

pc = Pinecone(api_key=pinecone_api_key)


########### Setting up Pinecone index
index = pc.Index("ragtest")

vectors = [
    (f"doc_{i}", embeddings[i].tolist(), {"text": texts[i]})
    for i in range(len(texts))
]


########### Injesting embeddings in the index
index.upsert(vectors=vectors)
print(f"Upserted {len(vectors)} vectors")

########### Function to do vector search in Pinecone
def vector_search_pinecone(query, top_k):
    print("*"*50)
    print("*"*50)
    print("*"*50)
    print("*"*50)
    print("vector search called")
    q_emb   = model.encode([query])[0].tolist()
    results = index.query(vector=q_emb, top_k=top_k, include_metadata=True)
    return [
        {"id": m["id"], "content": m["metadata"]["text"], "score": m["score"]}
        for m in results["matches"]
    ]

########### RAG with Pinecone
def rag_pinecone(query, top_k=3):
    """Plain vector RAG"""
    results = vector_search_pinecone(query, top_k)
    return _generate(query, results)

print(rag_pinecone(q))

Upserted 52 vectors
**************************************************
**************************************************
**************************************************
**************************************************
vector search called


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.99it/s]
2026-04-18 10:00:15,530 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The context provided does not directly answer the question "What is attention is all you need?" However, based on the information given, it appears that the question is referencing the title of a research paper, "Attention Is All You Need," which introduced the Transformer model.

In this context, "Attention Is All You Need" refers to the idea that attention mechanisms can be used as the primary component of a neural network architecture, replacing traditional recurrent neural network (RNN) and convolutional neural network (CNN) components. The paper argues that attention mechanisms are sufficient for machine translation tasks and can achieve state-of-the-art results without the need for RNNs or CNNs.

In essence, the answer to the question is that "Attention Is All You Need" is a research paper that proposes a new neural network architecture, the Transformer, which relies solely on attention mechanisms to achieve high-performance results in machine translation tasks.


### RAG Pinecone with metadata filtering

In [5]:
# ── Store with metadata ───────────────────────────────────────────────────────
vectors = [
    (
        f"doc_{i}",
        embeddings[i].tolist(),
        {
            "text":        texts[i],
            "source_file": chunks[i].metadata.get("source_file", ""),
            "page":        chunks[i].metadata.get("page", 0),
        }
    )
    for i in range(len(texts))
]
index.upsert(vectors=vectors)

# ── Vector search with filter ─────────────────────────────────────────────────
def vector_search_with_metadata_filtering(query, top_k, filter=None):
    q_emb   = model.encode([query])[0].tolist()
    results = index.query(vector=q_emb, top_k=top_k, include_metadata=True, filter=filter)
    return [
        {"id": m["id"], "content": m["metadata"]["text"], "score": m["score"]}
        for m in results["matches"]
    ]

query = "What is attention is all you need paper?"


# ── Usage examples ────────────────────────────────────────────────────────────
# Single file
vector_search_with_metadata_filtering(query, top_k=3, filter={"source_file": {"$eq": "attention.pdf"}})

# Specific pages
vector_search_with_metadata_filtering(query, top_k=3, filter={"page": {"$in": [1, 2, 3]}})

# Page range
vector_search_with_metadata_filtering(query, top_k=3, filter={"page": {"$gte": 5, "$lte": 10}})

# Multiple conditions (AND)
vector_search_with_metadata_filtering(query, top_k=3, filter={
    "source_file": {"$eq": "attention.pdf"},
    "page":        {"$gte": 5}
})


def rag_hybrid_with_meta_data(query, top_k=3, filter=None):
    results = rrf(
        bm25_search(query, top_k * 3),
        vector_search_with_metadata_filtering(query, top_k * 3, filter=filter)
    )[:top_k]
    return _generate(query, results)

# Usage
rag_hybrid_with_meta_data("What is attention?", filter={"source_file": {"$eq": "attention.pdf"}, "page": {"$in": [1, 2, 3]}})

Batches: 100%|██████████| 1/1 [00:00<00:00, 21.05it/s]
2026-04-18 10:00:20,240 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


'The context provided does not explicitly define what attention is. However, based on the information given, it appears that attention refers to a mechanism in neural networks, specifically in the context of natural language processing, that allows the model to focus on certain parts of the input data (such as words or phrases) when processing it.\n\nIn the context of the encoder self-attention mechanism mentioned, attention seems to refer to the ability of the model to learn and perform different tasks, such as anaphora resolution, by weighing the importance of different input elements. The attention heads mentioned appear to be components of the model that learn to attend to specific parts of the input data in order to accomplish these tasks.\n\nTherefore, a general definition of attention in this context could be: a mechanism in neural networks that allows the model to selectively focus on certain parts of the input data, weighing their importance, in order to perform specific tasks

## Semantic Caching

In [8]:
import numpy as np

# ── Semantic Cache ────────────────────────────────────────────────────────────
cache = []   # list of {"embedding": ..., "query": ..., "answer": ...}
CACHE_THRESHOLD = 0.90   # cosine similarity threshold, tune as needed

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def check_cache(query_emb):
    for entry in cache:
        if cosine_similarity(query_emb, entry["embedding"]) >= CACHE_THRESHOLD:
            return entry["answer"]
    return None

def add_to_cache(query_emb, answer):
    cache.append({"embedding": query_emb, "answer": answer})

# ── RAG with semantic cache ───────────────────────────────────────────────────
def rag_with_semantic_caching(query, top_k=3):
    query_emb = model.encode([query])[0]

    # Check cache first
    cached = check_cache(query_emb)
    if cached:
        print("Cache hit!")
        return cached

    # Cache miss — run normal RAG
    results = vector_search(query, top_k)
    answer  = _generate(query, results)

    add_to_cache(query_emb, answer)
    return answer

In [10]:
answer = rag_with_semantic_caching("What is attention is all you need?")
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 23.20it/s]

Cache hit!
The context provided doesn't explicitly answer the question "What is attention is all you need?" However, based on the information given, it appears to be referencing the paper "Attention Is All You Need" by Vaswani et al., which introduced the Transformer model.

In this context, "Attention Is All You Need" refers to the idea that attention mechanisms can be used as the primary component of a neural network architecture, replacing traditional recurrent neural network (RNN) and convolutional neural network (CNN) components. The Transformer model, which is described in the context, uses self-attention mechanisms to process input sequences, and it has been shown to be highly effective in various natural language processing tasks.

So, in essence, "Attention Is All You Need" refers to the concept that attention mechanisms can be sufficient for building powerful neural network models, without the need for other components like RNNs or CNNs.
